### **Agentic Search**

In [14]:
from dotenv import load_dotenv, find_dotenv
import os
from tavily import TavilyClient
from pprint import pprint

# load environment variables from .env file
_ = load_dotenv(find_dotenv())
# connect
client = TavilyClient(api_key = os.environ.get('TAVILY_API_KEY'))

In [3]:
# run search
response = client.search('What is in Nvidia\'s new Blackwell GPU?', include_answer = True)

# print the answer
response['answer']

'The Blackwell GPU features 208 billion transistors and offers up to 20 petaflops of compute. It includes advanced Tensor Cores and a Transformer Engine for AI.'

#### **Regular Search**

In [6]:
# choose location (try to change to your own city!)

city = 'San Francisco'

query = f'''
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
'''

In [ ]:
!pip install duckduckgo_search

import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def regular_search(query, max_results = 5):
    try:
        results = ddg.text(query, max_results = max_results)
        return [i['href'] for i in results]
    except Exception as e:
        print('returning previous results due to exception reaching ddg.')
        results = [  # cover case where DDG rate limits due to high deeplearning.ai volume
            'https://weather.com/weather/today/l/USCA0987:1:US',
            'https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8',
        ]
        return results  

for i in regular_search(query):
    print(i)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


https://weather.com/weather/today/l/San+Francisco+CA+USCA0987:1:US
https://weather.com/weather/tenday/l/San+Francisco+CA+USCA0987:1:US
https://weather.com/weather/today/l/San+Francisco+CA?canonicalCityId=45cf83277ba620e7dc8a0fe8b6eda89925a3e6d2e1bdfef3f74a1590017bd70d
https://weather.com/weather/tenday/l/San+Francisco+CA+USCA0987:1:US
https://weather.com/weather/tenday/l/San+Diego+CA?canonicalCityId=3b2b39ed755b459b725bf2a29c71d678


In [10]:
def scrape_weather_info(url):
    '''Scrape content from the given URL'''
    if not url:
        return 'Weather information could not be found.'
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers = headers)
    if response.status_code != 200:
        return 'Failed to retrieve the webpage.'

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

# use DuckDuckGo to find websites and take the first result
url = regular_search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f'Website: {url}\n\n')
print(str(soup)[:50000]) # limit long outputs

Website: https://weather.com/weather/today/l/San+Francisco+CA+USCA0987:1:US


Failed to retrieve the webpage.


In [ ]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(' ', strip = True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = '\n'.join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f'Website: {url}\n\n')
print(weather_data)


#### **Agentic Search**

In [15]:
# run search
response = client.search(query, max_results = 1)

# print first result
data = response['results'][0]['content']

pprint(data)

("{'location': {'name': 'San Francisco', 'region': 'California', 'country': "
 "'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': "
 "'America/Los_Angeles', 'localtime_epoch': 1748420088, 'localtime': "
 "'2025-05-28 01:14'}, 'current': {'last_updated_epoch': 1748419200, "
 "'last_updated': '2025-05-28 01:00', 'temp_c': 12.2, 'temp_f': 54.0, "
 "'is_day': 0, 'condition': {'text': 'Overcast', 'icon': "
 "'//cdn.weatherapi.com/weather/64x64/night/122.png', 'code': 1009}, "
 "'wind_mph': 8.7, 'wind_kph': 14.0, 'wind_degree': 251, 'wind_dir': 'WSW', "
 "'pressure_mb': 1017.0, 'pressure_in': 30.03, 'precip_mm': 0.0, 'precip_in': "
 "0.0, 'humidity': 86, 'cloud': 100, 'feelslike_c': 10.7, 'feelslike_f': 51.3, "
 "'windchill_c': 9.6, 'windchill_f': 49.3, 'heatindex_c': 11.0, 'heatindex_f': "
 "51.9, 'dewpoint_c': 10.5, 'dewpoint_f': 50.8, 'vis_km': 16.0, 'vis_miles': "
 "9.0, 'uv': 0.0, 'gust_mph': 12.8, 'gust_kph': 20.7}}")


In [ ]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent = 4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1748420088,
        "localtime": "2025-05-28 01:14"
    },
    "current": {
        "last_updated_epoch": 1748419200,
        "last_updated": "2025-05-28 01:00",
        "temp_c": 12.2,
        "temp_f": 54.0,
        "is_day": 0,
        "condition": {
            "text": "Overcast",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/122.png",
            "code": 1009
        },
        "wind_mph": 8.7,
        "wind_kph": 14.0,
        "wind_degree": 251,
        "wind_dir": "WSW",
        "pressure_mb": 1017.0,
        "pressure_in": 30.03,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 86,
        "cloud": 100,
        "feelslike_c": 10.7,
        "feelslike_f": 51.3,
        "windchill_c": 9.6,
       